In [ ]:
from typing import Callable, Tuple
import numpy as np

In [ ]:
def cuadratura_gauss_legendre(
    func: Callable[[np.ndarray], np.ndarray], n: int
) -> float:
    """Calcula la integral definida de una función en el intervalo [-1, 1]

    utilizando la cuadratura de Gauss-Legendre de n puntos.

    Este método es exacto para polinomios de grado hasta 2n - 1.

    Parámetros
    ----------
    func : Callable[[np.ndarray], np.ndarray]
        Función a integrar. Debe aceptar un arreglo de NumPy y devolver
        un arreglo de la misma forma (vectorizada).
    n : int
        Número de puntos de cuadratura (nodos) a utilizar. Debe ser >= 1.

    Devuelve
    -------
    float
        Aproximación numérica de la integral en [-1, 1].

    Lanza
    -----
    ValueError
        Si el número de puntos `n` es menor que 1.
    """
    if n < 1:
        raise ValueError("El número de puntos 'n' debe ser un entero positivo.")

    # Obtenemos los nodos (raíces del polinomio de Legendre P_n(x))
    # y los pesos correspondientes mediante el algoritmo de Golub-Welsch.
    nodos, pesos = np.polynomial.legendre.leggauss(n)

    # Evaluamos la función en los nodos y calculamos el producto punto con los pesos
    integral_aprox = np.sum(pesos * func(nodos))

    return float(integral_aprox)

In [ ]:
def nodos_pesos_manual(
    n: int, tol: float = 1e-14, max_iter: int = 100
) -> Tuple[np.ndarray, np.ndarray]:
    """Calcula los nodos y pesos de Gauss-Legendre desde cero usando

    la relación de recurrencia de Bonnet y el método de Newton-Raphson.

    Parámetros
    ----------
    n : int
        Grado del polinomio de Legendre (número de puntos).
    tol : float, opcional
        Tolerancia para la convergencia de Newton-Raphson (por defecto 1e-14).
    max_iter : int, opcional
        Número máximo de iteraciones permitidas (por defecto 100).

    Devuelve
    -------
    Tuple[np.ndarray, np.ndarray]
        Tupla con dos arreglos: (nodos, pesos).
    """
    # Aproximación inicial de las raíces (simétricas respecto a 0)
    i = np.arange(1, n + 1)
    x = np.cos(np.pi * (4 * i - 1) / (4 * n + 2))

    for _ in range(max_iter):
        # Evaluación del polinomio P_n(x) mediante recurrencia:
        # (k + 1) P_{k+1}(x) = (2k + 1) x P_k(x) - k P_{k-1}(x)
        p0 = np.ones_like(x)  # P_0(x) = 1
        p1 = x.copy()  # P_1(x) = x

        for k in range(1, n):
            p2 = ((2 * k + 1) * x * p1 - k * p0) / (k + 1)
            p0, p1 = p1, p2

        # p1 contiene ahora P_n(x) y p0 contiene P_{n-1}(x)
        # Derivada de P_n(x): P'_n(x) = n * (x * P_n(x) - P_{n-1}(x)) / (x^2 - 1)
        dp = n * (x * p1 - p0) / (x**2 - 1)

        # Paso de Newton-Raphson
        dx = p1 / dp
        x -= dx

        if np.max(np.abs(dx)) < tol:
            break

    # Cálculo de los pesos: w_i = 2 / ((1 - x_i^2) * [P'_n(x_i)]^2)
    pesos = 2.0 / ((1.0 - x**2) * dp**2)

    # Ordenamos los nodos de menor a mayor junto con sus pesos
    orden = np.argsort(x)
    return x[orden], pesos[orden]

In [ ]:
# =====================================================================
# Ejemplo de uso 
# =====================================================================
if __name__ == "__main__":
    # Ejemplo 1: Polinomio de grado 5: f(x) = 3x^5 - 2x^3 + x^2 - 4
    # Integral analítica en [-1, 1]:
    # [x^6/2 - x^4/2 + x^3/3 - 4x] de -1 a 1 = (1/3 - 4) - (-1/3 + 4) = -22/3 ≈ -7.333333333333333
    f_polinomio = lambda x: 3 * x**5 - 2 * x**3 + x**2 - 4
    valor_exacto_pol = -22.0 / 3.0

    # Con n = 3, Gauss-Legendre es exacto hasta grado 2(3) - 1 = 5
    resultado_pol = cuadratura_gauss_legendre(f_polinomio, n=3)
    print(f"--- Prueba Polinomio Grado 5 (n=3) ---")
    print(f"Exacto:    {valor_exacto_pol:.15f}")
    print(f"Calculado: {resultado_pol:.15f}")
    print(f"Error abs: {abs(valor_exacto_pol - resultado_pol):.2e}\n")

    # Ejemplo 2: Función trascendente: f(x) = exp(x) * cos(x)
    # Integral analítica en [-1, 1]: 0.5 * [exp(x)*(sin(x) + cos(x))] de -1 a 1
    f_trascendente = lambda x: np.exp(x) * np.cos(x)
    valor_exacto_tras = 0.5 * (
        np.exp(1) * (np.sin(1) + np.cos(1))
        - np.exp(-1) * (np.sin(-1) + np.cos(-1))
    )

    print(f"--- Prueba Función Trascendente exp(x)*cos(x) ---")
    for n_puntos in [2, 4, 6]:
        res = cuadratura_gauss_legendre(f_trascendente, n=n_puntos)
        error = abs(valor_exacto_tras - res)
        print(f"n = {n_puntos} -> Resultado: {res:.15f} | Error: {error:.2e}")